In [1]:
# the codes in this cell are not important

from multiprocessing import context
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# calculate the attention scores for the second input

query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)


# calculate the attention weights for the second input
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())


# calculate the context vector for the second input

query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape) 
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)


## explain the following lines of code

"""

1st line: attn_scores_2 = torch.empty(inputs.shape[0])

This line creates an empty tensor of shape (inputs.shape[0],) to store the attention scores for each input token.

2nd line: context_vec_2 = torch.zeros(query.shape) 

This line creates a tensor of shape (query.shape) to store the context vector for the second input.


"""

## for ALL INPUTS

attn_scores = inputs @ inputs.T

attn_weights = torch.softmax(attn_scores, dim=-1) # what is the last dimension? the last dimension is the number of input tokens.

context_vec = attn_weights @ inputs





/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
tensor([0.4419, 0.6515, 0.5683])


In [ ]:
print(attn_weights.shape)
#

torch.Size([6, 6])


In [3]:
## self-attention with trainable weights
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

NameError: name 'd_in' is not defined

## first method to apply casual attention (Not recommended)
to hide future words

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)



In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

masked_simple = attn_weights * mask_simple
print(masked_simple)

"""
However, if the mask were applied after softmax, like above, it would disrupt the probability distribution created by softmax
Softmax ensures that all output values sum to 1
Masking after softmax would require re-normalizing the outputs to sum to 1 again, which complicates the process and might lead to unintended effects
"""

row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)


## Second method to apply casual attention (recommended)
to hide future words



In [ ]:
"""
Instead of masking the attention scores after softmax, we can mask the attention scores before softmax
"""

mask = torch.triu(torch.ones(context_length,context_length), diagonal=1)
msked_attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)

attn_weights = torch.softmax(msked_attn_scores / keys.shape[-1]**0.5, dim=-1)

print(attn_weights)

## what context length means? it means the number of input tokens.

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch)
print("-"*100)
print(batch.shape)


tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
----------------------------------------------------------------------------------------------------
torch.Size([2, 6, 3])


In [ ]:
class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens_,d_in = x.shape
        
        queries =self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            mask.bool()[:num_tokens,:num_tokens],
            -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec
    

context_length = batch.shape[1]

ca = CasualAttention(d_in, d_out, context_length, dropout)

context_vec = ca(batch)



# multi head attn 

In [ ]:
# SImple way to do multi head attn

class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

In the implementation above, the embedding dimension is 4, because we d_out=2 as the embedding dimension for the key, query, and value vectors as well as the context vector. And since we have 2 attention heads, we have the output embedding dimension 2*2=4

- While the above is an intuitive and fully functional implementation of multi-head attention (wrapping the single-head attention CausalAttention implementation from earlier), we can write a stand-alone class called MultiHeadAttention to achieve the same


- We don't concatenate single attention heads for this stand-alone MultiHeadAttention class

- Instead, we create single W_query, W_key, and W_value weight matrices and then split those into individual matrices for each attention head

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # splits the output dim across the attention heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        """
        1. compute the query, key, and value matrices
        2. split the matrices into individual attention heads
        3. we transpose the query and key and value matrices 
        4. compute the attention scores
        5. do the masking
        6. compute the attention weights
        7. compute the context vectors
        8. concatenate the context vectors
        9. project the concatenated context vectors to the output dimension
        """
        b, num_tokens, d_out = x.shape

        # step 1
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # step 2
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # step 3
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # step 4
        attn_scores = queries @ keys.transpose(2,3)

        # step 5
        """ 
        Full mask (1024×1024):      Sliced mask (6×6):
        [0 1 1 1 ...]               [0 1 1 1 1 1]
        [0 0 1 1 ...]  → slice →    [0 0 1 1 1 1]
        [0 0 0 1 ...]               [0 0 0 1 1 1]
        ...                         [0 0 0 0 1 1]
        [0 0 0 0 ...]               [0 0 0 0 0 1]
        [0 0 0 0 ...]               [0 0 0 0 0 0]
        """ 
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
        attn_scores.masked_fill(mask_bool, -torch.inf)

        # step 6
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # step 7
        context_vec = (attn_weights @ values).transpose(1,2)

        # step 8 
        """ 
        1. view is to flatten the context vectors across the attention heads to shape (b, num_tokens, d_out)
        2. after transpose, heads are interleaved in memory,
        3. we can't use view here unless we make a contiguous copy of the tensor
        """
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # step 9
        """
        Linear layer to combine head outputs

        After concatenation:
        token: [head0_output | head1_output | head2_output | ...]
                ^^^ 64 dims    ^^^ 64 dims    ^^^ 64 dims

        Each head learned different patterns, but they don't interact yet. out_proj lets them communicate:

        After projection:
        
        out_proj:  [head0_info + head1_info + head2_info + ...] → blended representation

        What it does

        Linear transform: d_out × d_out matrix multiplication
        Cross-head mixing: Each output dimension can attend to any input head
        Scale control: Adjusts magnitude before residual connection + LayerNorm

        Why "optional"
        Technically the model works without it (heads already computed attention), but:

        Standard: Every major transformer (GPT, BERT, T5) uses it
        Performance: ~2-3% better with it (heads specialize better when they know they'll be mixed)

        Short: Heads work in parallel → out_proj combines their work.
        """
        context_vec = self.out_proj(context_vec)
        return context_vec
    
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vec = mha(batch)

print(context_vec)
print("context_vec.shape:", context_vec.shape)

Note that in addition, we added a linear projection layer (self.out_proj ) to the MultiHeadAttention class above. This is simply a linear transformation that doesn't change the dimensions. It's a standard convention to use such a projection layer in LLM implementation, but it's not strictly necessary (recent research has shown that it can be removed without affecting the modeling performance; see the further reading section at the end of this chapter)